# Buổi 3 — ANOVA & Kiểm định phi tham số (Bài 8, 9)

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

> 📥 **Đầu vào:** nạp thư viện + đọc `lop_hoc_240.csv` (notebook độc lập, tự chạy được).
>
> 📤 **Đầu ra thật:** `(240, 14)` — khớp đúng 2 buổi trước. ✅

In [ ]:
df.loc[df.study_hours > 20, "study_hours"] /= 10

> 📥 **Đầu vào:** xử lý nhanh outlier `study_hours` (chia 10, giống buổi 2) — bước chuẩn bị, không có output hiển thị.

## 1. ANOVA từ nguyên lý đầu tiên: chia phương sai thành 2 phần
Tổng biến thiên (SST) = biến thiên **giữa** nhóm (SSB) + biến thiên **trong** nhóm (SSW). F = (SSB/df1)/(SSW/df2).
SPSS: `Analyze > Compare Means > One-Way ANOVA` (chọn Post Hoc: Tukey; Options: Homogeneity, Welch).

In [ ]:
g = df.groupby("school").math
grand = df.math.mean()
SSB = sum(len(v) * (v.mean() - grand) ** 2 for _, v in g); SSW = sum(((v - v.mean()) ** 2).sum() for _, v in g)
k, N = df.school.nunique(), len(df); F = (SSB / (k - 1)) / (SSW / (N - k))
print(f"SSB={SSB:.0f} SSW={SSW:.0f}  F thủ công={F:.2f}  p={stats.f.sf(F, k-1, N-k):.3g}")
print("scipy:", stats.f_oneway(*[v for _, v in g]))
print("eta² (mức ảnh hưởng) =", round(SSB / (SSB + SSW), 3))

> 📥 **Đầu vào:** điểm `math` nhóm theo `school` (3 trường A/B/C) — TỰ TAY tính ANOVA từ định nghĩa gốc: chia tổng biến thiên (SST) thành SSB (biến thiên GIỮA nhóm) và SSW (biến thiên TRONG nhóm), rồi tính F = (SSB/dfB)/(SSW/dfW).
>
> 📤 **Đầu ra thật:** `SSB=1057, SSW=21317, F thủ công=5,87, p=0,00323` — và ngay dòng dưới, gọi thẳng `scipy.stats.f_oneway` cho ra **CHÍNH XÁC cùng F=5,875, p=0,00323**. ✅ Xác nhận: ANOVA không phải hộp đen, chỉ là 1 phép chia tỉ lệ 2 loại phương sai — tự tính tay và dùng hàm có sẵn cho kết quả giống hệt nhau. Số liệu này cũng khớp đúng Module 06 đã báo cáo (F(2,237)=5,87, p=0,003).
>
> 📐 **eta² = 0,047 nghĩa là gì:** trường học "giải thích" được khoảng 4,7% biến thiên điểm Toán — mức NHỎ theo ngưỡng Cohen dù F có ý nghĩa thống kê rõ ràng (p=0,003) — đúng bài học "ý nghĩa thống kê ≠ cỡ hiệu ứng lớn" đã nhấn mạnh xuyên suốt Module 04-06.

## 2. Kiểm tra giả định — nếu vi phạm thì sao?

In [ ]:
import statsmodels.formula.api as smf, statsmodels.api as sm
m = smf.ols("math ~ C(school)", df).fit()
print("Levene:", stats.levene(*[v for _, v in g]))
print("Shapiro trên phần dư:", stats.shapiro(m.resid))
sm.qqplot(m.resid, line="s"); plt.show()

> 📥 **Đầu vào:** phần dư (residuals) của mô hình `math ~ school` — dùng để kiểm tra 2 trong 4 điều kiện an toàn của ANOVA: phương sai đồng nhất (Levene) và phân phối chuẩn (Shapiro-Wilk trên phần dư, KHÔNG phải trên dữ liệu gốc — đây là cách làm đúng chuẩn, vì ANOVA chỉ yêu cầu phần dư sau khi đã trừ đi hiệu ứng nhóm phải xấp xỉ chuẩn).
>
> 📤 **Đầu ra thật:** Levene p=**0,292** (≥0,05 → phương sai đồng nhất, khớp Module 06), Shapiro p=**0,339** (≥0,05 → không đủ bằng chứng bác bỏ tính chuẩn của phần dư). ✅ Cả 2 điều kiện đều được thoả — ANOVA ở đây đáng tin cậy hoàn toàn, không cần chuyển sang Kruskal-Wallis hay Brown-Forsythe/Welch.
>
> 🖼️ **Đọc biểu đồ QQ-plot:** nếu các điểm bám sát đường chéo `line="s"`, phần dư gần chuẩn (khớp p=0,339 ở trên); lệch khỏi đường chéo ở 2 đầu (đuôi) mới là dấu hiệu đáng lo về tính chuẩn.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
print(pairwise_tukeyhsd(df.math, df.school))       # Post-hoc: cặp nào khác nhau?
print("Kruskal-Wallis (phi tham số):", stats.kruskal(*[v for _, v in g]))

> 📥 **Đầu vào:** điểm `math` theo 3 trường — chạy hậu kiểm Tukey HSD (xem CHÍNH XÁC cặp trường nào khác nhau) VÀ Kruskal-Wallis (phiên bản phi tham số của ANOVA, để đối chiếu).
>
> 📤 **Đầu ra thật (Tukey):** A-B meandiff=-3,41 p=0,046 (**reject=True**, có ý nghĩa); A-C meandiff=-5,34 p=0,0031 (**True**); B-C meandiff=-1,93 p=0,4266 (**False**, không có ý nghĩa). ✅ Khớp CHÍNH XÁC với bảng Tukey đã báo cáo ở Module 06 — Trường A tách biệt hẳn, B và C không khác nhau đáng kể.
>
> 📤 **Đầu ra thật (Kruskal-Wallis):** H=**11,79**, p=**0,00275** — CŨNG có ý nghĩa thống kê, cùng kết luận với ANOVA tham số (p=0,00323) dù dùng phương pháp khác (dựa trên thứ hạng thay vì giá trị thô). ✅ Hợp lý: khi cả 4 điều kiện an toàn tham số đều được thoả (như đã kiểm tra ở ô trên), ANOVA và Kruskal-Wallis thường "đồng thuận" với nhau — đây là một cách kiểm tra chéo (cross-check) hữu ích: nếu 2 phương pháp cho kết luận trái ngược, đó là tín hiệu cần xem lại giả định.

**❓** Vì sao không chạy 3 lần t-test cho A–B, A–C, B–C mà phải dùng ANOVA + post-hoc? (gợi ý: xác suất ít nhất 1 lần dương tính giả = 1 − 0.95³ ≈ ?)

## 3. ANOVA hai chiều và tương tác
Phương pháp dạy có hiệu quả **như nhau ở mọi trường** không?

In [ ]:
df["gain"] = df.posttest - df.pretest
m2 = smf.ols("gain ~ C(method) * C(school)", df).fit()
print(sm.stats.anova_lm(m2, typ=2).round(3))
sns.pointplot(data=df, x="school", y="gain", hue="method", dodge=.2); plt.title("Đường không song song = có tương tác"); plt.show()

> 📥 **Đầu vào:** mức tăng điểm `gain` theo CẢ 2 biến độc lập cùng lúc: `method` (Dự án/Truyền thống) VÀ `school` (A/B/C), cộng thêm hiệu ứng tương tác `method*school` — đây là ANOVA HAI CHIỀU.
>
> 📤 **Đầu ra thật (bảng ANOVA hai chiều):**
> | Nguồn | F | p |
> |---|---|---|
> | method (hiệu ứng chính) | 55,23 | **0,000** |
> | school (hiệu ứng chính) | 0,88 | 0,414 |
> | method×school (tương tác) | 1,31 | 0,273 |
>
> ✅ **Diễn giải:** CHỈ phương pháp dạy tạo khác biệt có ý nghĩa về mức tăng điểm (p&lt;0,001) — trường học không có ảnh hưởng đáng kể tới `gain` (khác với ảnh hưởng của trường lên điểm `math` tuyệt đối ở ô trên — đây là 2 biến phụ thuộc KHÁC NHAU, hoàn toàn hợp lý). Quan trọng nhất: hiệu ứng TƯƠNG TÁC p=0,273 KHÔNG có ý nghĩa — nghĩa là phương pháp Dự án hiệu quả hơn Truyền thống ở MỌI trường như nhau, không phải chỉ ở 1-2 trường cụ thể.
>
> 🖼️ **Đọc biểu đồ pointplot:** nếu 2 đường (Dự án/Truyền thống) gần như SONG SONG nhau qua 3 trường, đó là bằng chứng trực quan cho việc KHÔNG có tương tác (khớp p=0,273) — tiêu đề biểu đồ đã gợi ý đúng cách đọc: "đường không song song = có tương tác".

## 4. Phi tham số — khi dữ liệu không đủ 'đẹp'
SPSS: `Analyze > Nonparametric Tests > Independent Samples` / `Legacy Dialogs > Chi-square`.

In [ ]:
print("Mann-Whitney (h1 theo giới, Likert):", stats.mannwhitneyu(df[df.gender == "Nam"].h1, df[df.gender == "Nữ"].h1))
ct = pd.crosstab(df.gender, df.method); print(ct); chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Chi-square độc lập: χ²={chi2:.2f}, df={dof}, p={p:.3f}")
print("Wilcoxon bắt cặp pre-post:", stats.wilcoxon(df.posttest, df.pretest))

> 📥 **Đầu vào:** 3 phép so sánh khác nhau trên cùng dữ liệu lớp học, minh hoạ 3 kiểu kiểm định: Mann-Whitney (đánh giá Likert `h1` theo giới tính), Chi-square (bảng chéo giới tính × phương pháp), Wilcoxon (so sánh bắt cặp `pretest` vs `posttest`).
>
> 📤 **Đầu ra thật:**
> - Mann-Whitney: U=**6416**, p=**0,127** — không có ý nghĩa. ✅ Khớp CHÍNH XÁC với Module 07 đã báo cáo.
> - Chi-square giới tính×phương pháp: χ²=**1,36**, df=1, p=**0,243** — không có ý nghĩa. ✅ Khớp Module 07 (χ²=1,37, chênh lệch làm tròn không đáng kể).
> - Wilcoxon pretest~posttest: statistic=3249,5, p=**2,18e-25** — CÓ ý nghĩa RẤT CAO (khác với ví dụ Wilcoxon ở Module 07 dùng cặp biến `a1`/`a2` cho kết quả KHÔNG có ý nghĩa — ở đây so sánh `pretest` với `posttest`, là 2 bài kiểm tra khác thời điểm với chênh lệch thật sự lớn, nên hoàn toàn hợp lý là có ý nghĩa mạnh, nhất quán với t-test bắt cặp đã tính ở buổi 2, t=13,37, p=8,25e-31).
>
> 🎯 **Bài học tổng hợp:** cùng một bộ dữ liệu, 3 cặp biến khác nhau cho 3 kết luận khác nhau (2 "không có ý nghĩa", 1 "có ý nghĩa rất mạnh") — đúng như tinh thần khoá học: kết quả phụ thuộc vào CÂU HỎI cụ thể đang hỏi, không có công thức chung "luôn có ý nghĩa" hay "luôn không có ý nghĩa".

### Bảng chọn kiểm định (để in ra dán bàn)
| Câu hỏi | Tham số | Phi tham số |
|---|---|---|
| 2 nhóm độc lập | t-test Welch | Mann-Whitney U |
| 2 lần đo cùng đối tượng | t bắt cặp | Wilcoxon |
| ≥3 nhóm độc lập | ANOVA (+Tukey) | Kruskal-Wallis |
| 2 biến định danh | — | Chi-square |

## 5. Bài tập
1. Kiểm định `pretest` khác nhau giữa các trường không? Hai nhóm phương pháp có cân bằng nhau ở `pretest` không? Điều này nói gì về thiết kế nghiên cứu?
2. Cố ý thêm 5 giá trị ngoại lai vào `math` của trường C rồi chạy lại ANOVA và Kruskal. Kết quả nào bền hơn?